# Experiment A - Raw-Acceleration CNN Baseline

**Question:** how much character-discriminative information is available in the producer's raw acceleration channels for held-out users?

Strict protocol:

1. Read raw acceleration from `paddedSpikeIMU[..., 15:18]`.
2. Split by **user**, never by segment.
3. Fit normalization using **raw train valid samples only**.
4. Train `MaskAwareAccelerationCNN` from scratch.
5. Select the best epoch using validation balanced accuracy.
6. Evaluate raw held-out test users with the shared representation protocol.
7. Save the authoritative A checkpoint used by B for weights, split, class mapping, and normalization; C/D reuse its split/class mapping.

This notebook is intentionally thin. The executable experiment lives in `scripts/run_experiment_a.py`; reusable implementation lives in `snn/accel_reconstruction_eval/`.


In [ ]:
from __future__ import annotations

from dataclasses import replace
from pathlib import Path
import sys

import pandas as pd
import torch


def find_repository_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'snn').is_dir() and (candidate / 'scripts').is_dir():
            return candidate
    raise FileNotFoundError('Could not locate repository root containing snn/ and scripts/.')


REPOSITORY_ROOT = find_repository_root(Path.cwd())
SCRIPTS_DIR = REPOSITORY_ROOT / 'scripts'
if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from snn.accel_reconstruction_eval import experiment_a_config
from run_experiment_a import run_experiment_a

print('Repository root:', REPOSITORY_ROOT)
print('CUDA available:', torch.cuda.is_available())


## Paths and experiment controls


In [ ]:
DATASET_ROOT = Path('outputs/action0_rectified/low-pass/aligned-board-events')
OUTPUT_DIR = Path('notebooks/artifacts/acceleration_cnn_representation')

RANDOM_SEED = 12345
NUM_EPOCHS = 30
EARLY_STOPPING_PATIENCE = 10
BATCH_SIZE = 128
NUM_WORKERS = 0
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 0.0
USE_CLASS_WEIGHTS = False
GRAD_CLIP_NORM = None
MAX_TRAIN_BATCHES = None
USE_GPU = True


## Build the Experiment A configuration

The domain contract is fixed: raw train / raw validation / raw test, with normalization fitted on raw train valid samples only.


In [ ]:
config = experiment_a_config(output_dir=OUTPUT_DIR, random_seed=RANDOM_SEED)
config = replace(
    config,
    use_gpu=USE_GPU,
    loader=replace(config.loader, batch_size=BATCH_SIZE, num_workers=NUM_WORKERS),
    training=replace(
        config.training,
        num_epochs=NUM_EPOCHS,
        early_stopping_patience=EARLY_STOPPING_PATIENCE,
        learning_rate=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
        use_class_weights=USE_CLASS_WEIGHTS,
        grad_clip_norm=GRAD_CLIP_NORM,
        max_train_batches=MAX_TRAIN_BATCHES,
    ),
    evaluation=replace(config.evaluation, random_seed=RANDOM_SEED),
)
config.validate()
config.to_dict()


## Run Experiment A

`load -> user split -> raw normalization -> train -> best validation epoch -> embeddings -> evaluate -> save artifacts`


In [ ]:
run = run_experiment_a(
    root=DATASET_ROOT,
    repository_root=REPOSITORY_ROOT,
    output_dir=OUTPUT_DIR,
    config=config,
)


## Primary results


In [ ]:
display(run.summary.T.rename(columns={0: 'Experiment A'}))


## CNN classification across splits


In [ ]:
display(run.classification_splits)


## Split contract


In [ ]:
display(run.split_summary)
display(run.label_split_counts)


## Raw-train normalization


In [ ]:
pd.DataFrame({
    'channel': ['x', 'y', 'z'],
    'mean': run.normalization.mean,
    'std': run.normalization.std,
}).assign(
    fitted_on=run.normalization.fitted_on,
    valid_time_points=run.normalization.valid_time_points,
)


## Training history


In [ ]:
display(run.training_history.tail(20))


## Saved artifacts


In [ ]:
artifact_table = pd.DataFrame([
    {'artifact': name, 'path': str(path)}
    for name, path in sorted(run.artifact_paths.items())
])
display(artifact_table)


## Interpretation

Experiment A is the raw-input reference condition. Interpret B/C/D relative to this run only when they reuse the same user split, class mapping, CNN architecture, and shared metric implementation.
